# HW3 Overview


In this assignment, we will study representations by vector space model, language model and neural networks, including TF-IDF and word embedding by word2vec.

We will reuse the same Yelp dataset and refer to each individual user review as a **document**. You should reuse your JSON parser in this assignment.

The same pre-processing steps you have developed in HW1 will be used in this assignment, i.e., tokenization, stemming, normalization and stopword removal. 

# Word Embedding (50 points)

Use [genism](https://radimrehurek.com/gensim/models/word2vec.html) library, download a pre-trained model or train a word2vec model (10 epochs) using the review data. Then use the model to get the vector representations of the document and query.

Hint: You can average the embedding of the terms to get the vector representation 
for a document.

Use the following 10 queries (same as the ones in HW1) and retrieve the top 3 documents for each query based on cosine similarity:

	general chicken
	fried chicken
	BBQ sandwiches
	mashed potatoes
	Grilled Shrimp Salad
	lamb Shank
	Pepperoni pizza
	brussel sprout salad
	FRIENDLY STAFF
	Grilled Cheese

Note that the training does not require GPU -- moderate CPU is good enough.  If your computation power is limited, i.e., limited memory or cpu, you can choose to download a pre-trained model or train the model on partial data, e.g., 50% or fewer review data. Please document the corresponding training detail in your report.

**What to submit**:

1. Paste your implementation of document representation and cosine similarity calculation. Report the training time of the word embedding model.
2. For the top 3 documents of each query, print the document and its cosine similarity score.
3. For the first three queries, analyze the relation between relevance and cosine similarity score: is a high score document more relevant to the query?



In [1]:
import json
import nltk
import re
import string
import os
import numpy as np

yelp_restaurants = []
stop_words = set(nltk.corpus.stopwords.words('english'))
TTF = {}
bigram_count = {}
total_num_words = 0
dataset = []
dataset_content = []
for yelp_fp in os.listdir('./yelp/'):
    with open('./yelp/' + yelp_fp, "r") as restaurant:
        yelp_restaurants.append(json.load(restaurant))


for restaurant in yelp_restaurants:
    for review in restaurant['Reviews']:
        # freq = {}
        # Normalize before tokenization to prevent empty tokens e.g."$ 1.1" -> ['', '11']
        tokens = re.sub(r"\b\d+\b", "#", review['Content']
            .translate(str.maketrans('', '', string.punctuation))).lower()

        # Tokenize by word
        tokens = nltk.word_tokenize(tokens)

        # # Stopword removal
        tokens = [word for word in tokens if not word in stop_words]

        # Token stemming
        tokens = [nltk.stem.PorterStemmer().stem(token) for token in tokens]

        # Write back to the object
        review['NormalizedContent'] = tokens

        # Store in dataset for word2vec/tfidf model
        dataset.append(review['NormalizedContent'])
        dataset_content.append(review['Content'])


In [2]:
import time
from gensim.models import Word2Vec


start = time.time()
model = Word2Vec(sentences=dataset, min_count=1, workers=4, epochs=10)
end = time.time()
print(f"Word embedding model training time: {end - start}")
model.save("word2vec.model")

Word embedding model training time: 28.406258821487427


In [6]:
import numpy as np

def cosine_similarity(V_q,V_d):
    return V_q @ (V_d/ np.linalg.norm(V_d))

queries = [
    "general chicken", 
    "fried chicken",
    "BBQ sandwiches",
    "mashed potatoes",
    "Grilled Shrimp Salad",
    "lamb Shank",
    "Pepperoni pizza",
    "brussel sprout salad",
    "FRIENDLY STAFF",
    "Grilled Cheese"
]
processed_queries = []
for query in queries:
    # Tokenize by word
    tokens = nltk.word_tokenize(query)
    # # Stopword removal
    tokens = [word for word in tokens if not word in stop_words]
    # Token stemming
    tokens = [nltk.stem.PorterStemmer().stem(token) for token in tokens]
    processed_queries.append(tokens)

# Compute all doc V_d
for restaurant in yelp_restaurants:
    for review in restaurant['Reviews']:
        if len(review['NormalizedContent']) != 0:
            review['normalized_wv'] = np.mean([model.wv[word] for word in review['NormalizedContent']], axis=0)

print("Top 3 document for each query:")
for query in processed_queries:
    doc_similarity = []
    V_q = np.mean([model.wv[word] for word in query], axis=0)
    for restaurant in yelp_restaurants:
        for review in restaurant['Reviews']:
            if len(review.get('normalized_wv', [])) != 0:
            # if review.get('normalized_wv', None) != None:
                doc_similarity.append((np.absolute(cosine_similarity(V_q, review['normalized_wv'])), review['Content']))
    print(f'query: {query}')
    for similarity, doc in sorted(doc_similarity, reverse=True)[:3]:
        print(f'Similarity: {similarity}')
        print(doc)

Top 3 document for each query:
query: ['gener', 'chicken']
Similarity: 0.0002946034073829651
Attempted to have dinner here last Friday (June 24). I was treating a friend, who had never been to this restaurant, to dinner. We arrived at 6:55pm and were informed it was a 20 minute wait for our party of 2--Not too bad, given it gets crowded. 30 minutes later we were told we could move into the inside waiting area and we were the next party of 2. After 45 minutes, I checked and was again informed we were the next party of 2. After an hour and seeing 3 different parties of 2 being seated before us I again confronted the host. He stated he informed me upfront the wait was about 20 minutes and the parties being seated were in front of us. (I actually saw two of the couples get out of their cabs, about 15 minutes into our wait) I unsure when about 20 minutes = an hour + but we decided to leave. Exceptionally disrespectful!!  Plenty of other great restaurants in Chicago.
Similarity: 0.0009553655

# TF-IDF (50 points)
Use the [gensim TF-IDF](https://radimrehurek.com/gensim/models/tfidfmodel.html) to estimate a TF-IDF model. Represent the document and query by TF-IDF vector and repeat the tasks in Word Embedding section.

**What to submit**:
1. Paste your implementation of document representation. Report the training time of the model.
2. For the top 3 documents of each query, print the document and its cosine similarity score.
3. For the first three queries, analyze the relation between relevance and cosine similarity score: is a high score document more relevant to the query?
4. Compare TF-IDF with averaged word2vec, which model gives a better document representation and query representation? Why?

In [4]:
from gensim.models import TfidfModel
from gensim.corpora import Dictionary

dct = Dictionary(dataset)  # fit dictionary
corpus = [dct.doc2bow(line) for line in dataset]

start = time.time()
tfidf_model = TfidfModel(corpus)
end = time.time()
print(f"TfidfModel training time: {end - start}")
tfidf_model.save("tfidf_model.model")

TfidfModel training time: 0.8700037002563477


In [7]:
from gensim import similarities

tfidf_corpus = tfidf_model[corpus]
print("Top 3 document for each query:")
for query in processed_queries[:1]:
    corpus_idx = 0
    doc_similarity = []
    query_bow = dct.doc2bow(query)
    index = similarities.SparseMatrixSimilarity(tfidf_corpus, num_features=len(dct))
    similarity_scores = index[tfidf_model[query_bow]]
    print(f'query: {query}')
    for idx, similarity in sorted(enumerate(similarity_scores), key=lambda x: x[1], reverse=False)[:3]:
        print(f'Similarity: {similarity}')
        print(dataset_content[idx])

Top 3 document for each query:
query: ['gener', 'chicken']
Similarity: 0.0
We had our very last meal in N.O here and it was great.  Their sweet tea was nice and sweet, and my meal was great.  I decided to try a]the craw fish pie with a side of greens.  It was pretty much an empanada stuffed with a filling similar to craw fish ettoufee.  the pastry was flaky and perfectly baked or fried, I am not even sure, but it was very good.  I tasted a bit of my friends macaroni and cheese, which was wonderful.  After lunch, we also decided to order dessert, which was the Peanut Brittle homemade ice cream.  I have never had ice cream so incredibly creamy.  It was definitely homemade and the complete opposite of ice cream out of a carton.  We loved it so much.  We also realized that there was a butcher shop located right next door that was also affiliated with Cachon.  We stopped there after finishing up our ice cream so that my friend's mother could order a traditional Muffaletta sandwich to take o

# Extra Credits (5pts)

You are encouraged to further investigate other document representation methods, e.g., [doc2vec](https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.htm). Is it harder to train? Does it perform better than the two models we investigated?

# Submission

This assignment has in total 100 points. The deadline is June 9th 23:59 PDT. You should submit your report in **PDF** using the homework latex template and submit your code (notebook). 